# Qwen3-4B-VHE-LoRA

Fine-tune Qwen2.5-7B for event extraction with standard LoRA on an unquantized FP16 base model. Every epoch is evaluated on the same DEV set using unconstrained greedy decoding.

**Configurable inputs:**
- `model_path`: attached base model path.
- `train.jsonl` and `dev.jsonl`: dataset paths.
- `RESUME_FROM_CHECKPOINT`: start from scratch or resume training.

> The best baseline adapter is selected on DEV. TEST is never used for checkpoint selection.

Train Qwen3-4B-VHE-LoRA and evaluate all DEV examples after every epoch.

## 1. Install dependencies

Standard LoRA does not use 4-bit quantization or require bitsandbytes.

In [1]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


## 2. Imports

Import model, dataset, LoRA, and evaluation utilities.

In [2]:
from pathlib import Path
import json
import shutil
import gc
import sys
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    set_seed,
    Trainer,
    TrainerCallback,
)
from transformers.trainer_utils import EvalLoopOutput

from peft import (
    get_peft_model,
    LoraConfig,
)

SEED = 42
set_seed(SEED)


## 3. Load tokenizer and base model

Load the Qwen2.5-7B base model attached to the Kaggle notebook in FP16 without quantization.

### LoRA conversion notes

- This copy loads the base model in FP16 instead of 4-bit NF4 and removes the k-bit preparation step.
- The `LoraConfig`, optimizer/training arguments, data, evaluation, and early-stopping settings are unchanged from the source notebook.
- The output directory has a `_lora` suffix so this run does not overwrite QLoRA checkpoints, predictions, or callback state.

In [3]:
# model_candidates = [
#     Path("/kaggle/input/models/qwen-lm/qwen2.5/transformers/7b/1"),
#     Path("/kaggle/input/qwen2.5/transformers/7b/1"),
# ]
# model_path = next((path for path in model_candidates if path.exists()), None)
# if model_path is None:
#     raise FileNotFoundError(
#         "Qwen2.5-7B was not found in any expected Kaggle model mount: "
#         + ", ".join(str(path) for path in model_candidates)
#     )

model_path = Path("/kaggle/input/models/qwen-lm/qwen-3/transformers/4b/1")
print("Model path:", model_path)

tokenizer = AutoTokenizer.from_pretrained(model_path)

# Load the unquantized base model in FP16.
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto",
)

# Required for gradient checkpointing during training.
model.config.use_cache = False

print("Model loaded!")
print("Device:", model.device)

Model path: /kaggle/input/models/qwen-lm/qwen-3/transformers/4b/1


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Model loaded!
Device: cuda:0


## 4. Tokenize one example

Concatenate input and target tokens while computing loss only on the target and EOS token.

In [4]:
def tokenize_example(example):
    input_text = example["input_text"]
    target_text = example["target_text"]

    input_tokens = tokenizer(
        input_text,
        add_special_tokens=True,
    )

    target_tokens = tokenizer(
        target_text,
        add_special_tokens=False,
    )

    input_ids = input_tokens["input_ids"]
    target_ids = target_tokens["input_ids"]

    input_ids = (
        input_tokens["input_ids"]
        + target_ids
        + [tokenizer.eos_token_id]
    )

    labels = (
        [-100] * len(input_tokens["input_ids"])
        + target_ids
        + [tokenizer.eos_token_id]
    )

    return {
        "input_ids": input_ids,
        "labels": labels,
        "attention_mask": [1] * len(input_ids),
    }

## 5. Format event-extraction data

Convert every sample into an `input_text` / JSON `target_text` pair.

In [5]:
def format_example(example):
    text = example["text"]

    target = {
        "events": example["events"]
    }

    return {
        "input_text": text,
        "target_text": json.dumps(
            target,
            ensure_ascii=False,
            separators=(",", ":")
        )
    }

## 6. Data collator

Pad each batch dynamically and mask label padding with `-100`.

In [6]:
def ee_data_collator(features):
    max_length = max(len(f["input_ids"]) for f in features)

    input_ids = []
    attention_mask = []
    labels = []

    for f in features:
        pad_length = max_length - len(f["input_ids"])

        input_ids.append(
            f["input_ids"] + [tokenizer.pad_token_id] * pad_length
        )

        attention_mask.append(
            f["attention_mask"] + [0] * pad_length
        )

        labels.append(
            f["labels"] + [-100] * pad_length
        )

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }

## 7. Configure and apply LoRA

Attach LoRA adapters to the attention and MLP projections.

In [7]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, peft_config)

## 8. Load and preprocess the dataset

Load the BKEE TRAIN and DEV JSONL files, format them, and tokenize the loss-evaluation copies.

In [8]:
# dataset = load_dataset(
#     "json",
#     data_files={
#         "train": "## Data path/train.jsonl",
#         "dev": "## Data path/dev.jsonl",
#     },
# )

dataset = load_dataset(
    "json",
    data_files={
        "train": "/kaggle/input/datasets/md0612/vhe-adapted/train.jsonl",
        "dev": "/kaggle/input/datasets/md0612/vhe-adapted/dev.jsonl",
    },
)

# Raw datasets
train_raw = dataset["train"]
dev_raw = dataset["dev"]

# Formatted datasets
train_formatted = train_raw.map(format_example)
dev_formatted = dev_raw.map(format_example)

# Tokenized datasets for training
train_dataset = train_formatted.map(
    tokenize_example,
    remove_columns=train_formatted.column_names,
)

# Tokenized dataset for validation loss.
dev_dataset = dev_formatted.map(
    tokenize_example,
    remove_columns=dev_formatted.column_names,
)

# Raw/formatted dataset for generation evaluation
dev_generation_dataset = dev_formatted

Generating train split: 0 examples [00:00, ? examples/s]

Generating dev split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/3267 [00:00<?, ? examples/s]

Map:   0%|          | 0/406 [00:00<?, ? examples/s]

Map:   0%|          | 0/3267 [00:00<?, ? examples/s]

Map:   0%|          | 0/406 [00:00<?, ? examples/s]

## 9. DEV metrics

After every epoch, generate unconstrained greedy predictions on the full DEV set in batches of four and compute TI, TC, AI, AC, and average F1.

In [9]:
# sys.path.insert(0, "## Metrics path")

sys.path.insert(0, "/kaggle/input/datasets/md0612/metrics")

from metrics.evaluator import Evaluator

metric_calculator = Evaluator()

## Dual-decoding trainer and checkpoint selector

Use deterministic greedy decoding without constrained decoding. Event types, roles, text, and spans remain model decisions.

In [10]:
EVENT_OUTPUT_SCHEMA = {
    "type": "object",
    "properties": {
        "events": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "event_type": {"type": "string"},
                    "trigger": {
                        "type": "array",
                        "minItems": 1,
                        "items": {
                            "type": "object",
                            "properties": {
                                "text": {"type": "string"},
                                "span": {
                                    "type": "array",
                                    "items": {"type": "integer"},
                                    "minItems": 2,
                                    "maxItems": 2,
                                },
                            },
                            "required": ["text", "span"],
                            "additionalProperties": False,
                        },
                    },
                    "arguments": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "role": {"type": "string"},
                                "mentions": {
                                    "type": "array",
                                    "items": {
                                        "type": "object",
                                        "properties": {
                                            "text": {"type": "string"},
                                            "span": {
                                                "type": "array",
                                                "items": {"type": "integer"},
                                                "minItems": 2,
                                                "maxItems": 2,
                                            },
                                        },
                                        "required": ["text", "span"],
                                        "additionalProperties": False,
                                    },
                                },
                            },
                            "required": ["role", "mentions"],
                            "additionalProperties": False,
                        },
                    },
                },
                "required": ["event_type", "trigger", "arguments"],
                "additionalProperties": False,
            },
        },
    },
    "required": ["events"],
    "additionalProperties": False,
}

RUN_ROOT = Path("/kaggle/working/ee_qwen2_5_7b_bkee_lora")
PREDICTION_ROOT = RUN_ROOT / "dev_predictions"
PREDICTION_ROOT.mkdir(parents=True, exist_ok=True)


class BaselineGenerationTrainer(Trainer):
    def __init__(
        self,
        generation_dataset,
        tokenizer,
        evaluator,
        prediction_root,
        max_new_tokens=256,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.generation_dataset = generation_dataset
        self.tokenizer = tokenizer
        self.evaluator = evaluator
        self.prediction_root = Path(prediction_root)
        self.max_new_tokens = max_new_tokens

    def _generate_baseline(self):
        predictions, golds, records = [], [], []
        tokenizer = self.tokenizer
        previous_padding_side = tokenizer.padding_side
        tokenizer.padding_side = "left"
        batch_size = DEV_PREDICT_BATCH_SIZE
        try:
            with torch.inference_mode():
                for index in range(0, len(self.generation_dataset), batch_size):
                    end = min(index + batch_size, len(self.generation_dataset))
                    examples = [self.generation_dataset[i] for i in range(index, end)]
                    inputs = tokenizer(
                        [example["input_text"] for example in examples],
                        padding=True,
                        return_tensors="pt",
                        add_special_tokens=True,
                    ).to(self.model.device)
                    generated_ids = self.model.generate(
                        **inputs,
                        max_new_tokens=self.max_new_tokens,
                        do_sample=False,
                        num_beams=1,
                        eos_token_id=tokenizer.eos_token_id,
                        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
                    )
                    decoded = tokenizer.batch_decode(
                        generated_ids[:, inputs.input_ids.shape[1]:],
                        skip_special_tokens=True,
                        clean_up_tokenization_spaces=False,
                    )
                    for offset, (example, prediction) in enumerate(zip(examples, decoded)):
                        gold = {"events": example["events"]}
                        predictions.append(prediction)
                        golds.append(gold)
                        records.append({
                            "index": index + offset,
                            "text": example["text"],
                            "gold": gold,
                            "prediction": prediction,
                        })
        finally:
            tokenizer.padding_side = previous_padding_side

        epoch = int(round(self.state.epoch or 0))
        output_path = self.prediction_root / f"epoch-{epoch}-baseline.jsonl"
        with output_path.open("w", encoding="utf-8") as handle:
            for record in records:
                handle.write(json.dumps(record, ensure_ascii=False) + "\n")

        results = self.evaluator.evaluate_on_all_sample(predictions, golds)
        valid_count = 0
        for prediction in predictions:
            try:
                parsed = json.loads(prediction)
                valid_count += int(
                    isinstance(parsed, dict)
                    and isinstance(parsed.get("events"), list)
                )
            except (json.JSONDecodeError, TypeError):
                pass
        results["valid_json_rate"] = valid_count / max(1, len(predictions))
        return results

    def evaluation_loop(
        self,
        dataloader,
        description,
        prediction_loss_only=None,
        ignore_keys=None,
        metric_key_prefix="eval",
    ):
        output = super().evaluation_loop(
            dataloader,
            description,
            True,
            ignore_keys,
            metric_key_prefix,
        )
        was_training = self.model.training
        old_use_cache = self.model.config.use_cache
        self.model.eval()
        self.model.config.use_cache = True
        try:
            baseline_results = self._generate_baseline()
        finally:
            self.model.config.use_cache = old_use_cache
            if was_training:
                self.model.train()

        metrics = dict(output.metrics)
        for mode, results in {"baseline": baseline_results}.items():
            metrics[f"{metric_key_prefix}_{mode}_avg_f1"] = results["avg_f1"]
            metrics[f"{metric_key_prefix}_{mode}_valid_json_rate"] = (
                results["valid_json_rate"]
            )
            for name in ("TI", "TC", "AI", "AC"):
                metrics[f"{metric_key_prefix}_{mode}_{name.lower()}_f1"] = (
                    results[name]["f1"]
                )
        self.baseline_generation_results = baseline_results
        return EvalLoopOutput(
            predictions=output.predictions,
            label_ids=output.label_ids,
            metrics=metrics,
            num_samples=output.num_samples,
        )


class BaselineEarlyStoppingCallback(TrainerCallback):
    def __init__(
        self,
        output_root,
        tokenizer,
        patience=3,
        threshold=0.002,
        grace_epochs=2,
        initial_state=None,
    ):
        self.output_root = Path(output_root)
        self.tokenizer = tokenizer
        self.patience = patience
        self.threshold = threshold
        self.grace_epochs = grace_epochs
        self.best = {"baseline": float("-inf")}
        self.early_stop_reference = {
            "baseline": float("-inf"),
        }
        self.best_epoch = {"baseline": None}
        self.stale_epochs = {"baseline": 0}
        if initial_state is not None:
            self.restore_state(initial_state)

    def restore_state(self, payload):
        if not isinstance(payload, dict):
            raise TypeError("Callback state must be a JSON object")

        expected_config = {
            "patience": self.patience,
            "threshold": self.threshold,
            "grace_epochs": self.grace_epochs,
        }
        for name, expected in expected_config.items():
            if payload.get(name) != expected:
                raise ValueError(
                    f"Callback state {name}={payload.get(name)!r} "
                    f"does not match current value {expected!r}"
                )

        for name in (
            "best_avg_f1",
            "best_epoch",
            "early_stop_reference",
            "stale_epochs",
        ):
            values = payload.get(name)
            if not isinstance(values, dict) or "baseline" not in values:
                raise ValueError(
                    f"Callback state is missing {name}['baseline']"
                )

        self.best["baseline"] = float(payload["best_avg_f1"]["baseline"])
        self.early_stop_reference["baseline"] = float(
            payload["early_stop_reference"]["baseline"]
        )
        best_epoch = payload["best_epoch"]["baseline"]
        self.best_epoch["baseline"] = (
            None if best_epoch is None else int(best_epoch)
        )
        self.stale_epochs["baseline"] = int(payload["stale_epochs"]["baseline"])

    def _save_state(self):
        payload = {
            "best_avg_f1": self.best,
            "best_epoch": self.best_epoch,
            "early_stop_reference": self.early_stop_reference,
            "stale_epochs": self.stale_epochs,
            "patience": self.patience,
            "threshold": self.threshold,
            "grace_epochs": self.grace_epochs,
        }
        self.output_root.mkdir(parents=True, exist_ok=True)
        (self.output_root / "best_state.json").write_text(
            json.dumps(payload, indent=2),
            encoding="utf-8",
        )

    def on_evaluate(self, args, state, control, metrics=None, model=None, **kwargs):
        if not state.is_world_process_zero:
            return control

        epoch = int(round(state.epoch or 0))
        for mode in ("baseline",):
            key = f"eval_{mode}_avg_f1"
            if metrics is None or key not in metrics:
                raise KeyError(f"Missing DEV selection metric: {key}")
            score = float(metrics[key])
            if score > self.best[mode]:
                self.best[mode] = score
                self.best_epoch[mode] = epoch
                best_dir = self.output_root / f"best-{mode}"
                model.save_pretrained(best_dir)
                self.tokenizer.save_pretrained(best_dir)

            significant_improvement = (
                score > self.early_stop_reference[mode] + self.threshold
            )
            if significant_improvement:
                self.early_stop_reference[mode] = score
                self.stale_epochs[mode] = 0
            elif epoch > self.grace_epochs:
                self.stale_epochs[mode] += 1
            else:
                self.stale_epochs[mode] = 0

        self._save_state()
        if (
            epoch > self.grace_epochs
            and self.stale_epochs["baseline"] >= self.patience
        ):
            control.should_training_stop = True
        return control

## Experiment configuration

Use full DEV after each epoch. Select the highest DEV average F1. The first two epochs can set the best score and do not consume early-stopping patience.

In [11]:
# Generation on the full DEV split
DEV_PREDICT_BATCH_SIZE = 4
DEV_MAX_NEW_TOKENS = 256

# Early stopping on full DEV average F1
EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_THRESHOLD = 0.002
EARLY_STOPPING_GRACE_EPOCHS = 2

training_args = TrainingArguments(
    output_dir=str(RUN_ROOT / "checkpoints"),

    num_train_epochs=10,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    logging_strategy="steps",
    logging_steps=25,
    logging_first_step=True,
    learning_rate=2e-4,
    warmup_steps=0,
    seed=SEED,
    lr_scheduler_type="constant",

    eval_strategy="epoch",
    save_strategy="steps",
    save_steps=50,
    save_total_limit=4,

    load_best_model_at_end=False,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=True,
    report_to="none",
)


## 11. Clear caches and inspect GPU memory

Clear Python/CUDA caches and print memory usage for every visible GPU.

In [12]:
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

for i in range(torch.cuda.device_count()):
    print(
        f"GPU {i}: "
        f"{torch.cuda.memory_allocated(i) / 1024**3:.2f} GB allocated, "
        f"{torch.cuda.memory_reserved(i) / 1024**3:.2f} GB reserved"
    )

GPU 0: 4.36 GB allocated, 4.38 GB reserved
GPU 1: 3.25 GB allocated, 3.28 GB reserved


## 12. Initialize the trainer and verify DEV batching

Use the same full DEV set for validation loss and baseline generation.

In [13]:
class SaveEpochCheckpoint(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        control.should_save = True
        return control


baseline_selection_callback = BaselineEarlyStoppingCallback(
    output_root=RUN_ROOT,
    tokenizer=tokenizer,
    patience=EARLY_STOPPING_PATIENCE,
    threshold=EARLY_STOPPING_THRESHOLD,
    grace_epochs=EARLY_STOPPING_GRACE_EPOCHS,
)

trainer = BaselineGenerationTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    generation_dataset=dev_generation_dataset,
    tokenizer=tokenizer,
    evaluator=metric_calculator,
    prediction_root=PREDICTION_ROOT,
    max_new_tokens=DEV_MAX_NEW_TOKENS,
    data_collator=ee_data_collator,
    callbacks=[baseline_selection_callback, SaveEpochCheckpoint()],
)

batch = next(iter(trainer.get_eval_dataloader()))
print("DEV samples:", len(dev_dataset))
print("DEV batch shape:", tuple(batch["input_ids"].shape))

# Fail-fast generation smoke test before spending time on epoch 1.
smoke_example = dev_generation_dataset[0]
smoke_inputs = tokenizer(
    smoke_example["input_text"],
    return_tensors="pt",
    add_special_tokens=True,
)
smoke_inputs = {
    key: value.to(model.device)
    for key, value in smoke_inputs.items()
}
old_use_cache = model.config.use_cache
model.eval()
model.config.use_cache = True
try:
    with torch.inference_mode():
        model.generate(
            **smoke_inputs,
            max_new_tokens=4,
            do_sample=False,
            num_beams=1,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
finally:
    model.config.use_cache = old_use_cache
    model.train()

print("PASS: baseline generation smoke test")

DEV samples: 406
DEV batch shape: (1, 191)
PASS: baseline generation smoke test


The run records DEV predictions and the best baseline adapter.

In [14]:
print(
    "Baseline early stopping:",
    f"patience={EARLY_STOPPING_PATIENCE},",
    f"threshold={EARLY_STOPPING_THRESHOLD},",
    f"grace_epochs={EARLY_STOPPING_GRACE_EPOCHS}",
)

Baseline early stopping: patience=3, threshold=0.002, grace_epochs=2


## 13. Train from the pretrained model

### Restore callback status

Set `CALLBACK_STATE_DIR` to the Kaggle Input directory containing both `best_state.json` and `best-baseline/` only when continuing a previous **LoRA** run. Leave it as `None` for a fresh LoRA run; do not load callback state or adapters from the QLoRA run.

This cell validates the saved early-stopping settings, restores the best score/epoch, improvement reference, and stale-epoch count, then copies the previous best adapter into writable `/kaggle/working`. Use files saved from the same run and corresponding to the checkpoint you resume. This does not resume model/optimizer progress; use your Trainer checkpoint resume argument separately.

In [15]:
# Example: Path("/kaggle/input/my-training-output/ee_qwen2_5_7b_bkee")
CALLBACK_STATE_DIR = None

if CALLBACK_STATE_DIR is not None:
    callback_state_dir = Path(CALLBACK_STATE_DIR)
    callback_state_path = callback_state_dir / "best_state.json"
    best_adapter_source = callback_state_dir / "best-baseline"

    if not callback_state_path.is_file():
        raise FileNotFoundError(f"Callback state not found: {callback_state_path}")
    if not best_adapter_source.is_dir():
        raise FileNotFoundError(f"Best adapter not found: {best_adapter_source}")

    callback_state = json.loads(callback_state_path.read_text(encoding="utf-8"))
    baseline_selection_callback.restore_state(callback_state)
    shutil.copytree(
        best_adapter_source,
        RUN_ROOT / "best-baseline",
        dirs_exist_ok=True,
    )
    print(
        "Restored callback state:",
        f"best_epoch={baseline_selection_callback.best_epoch['baseline']},",
        f"best_avg_f1={baseline_selection_callback.best['baseline']:.6f},",
        f"stale_epochs={baseline_selection_callback.stale_epochs['baseline']}",
    )
else:
    print("No callback state configured; starting callback state from scratch.")

No callback state configured; starting callback state from scratch.


In [16]:
trainer.train()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

## 14. Report selected adapters and outputs

The callback has already saved the selected baseline adapter. Save the final training state separately for recovery.

In [ ]:
FINAL_MODEL_DIR = RUN_ROOT / "final-training-state"
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print("Best baseline epoch:", baseline_selection_callback.best_epoch["baseline"])
print("Best baseline DEV avg_f1:", baseline_selection_callback.best["baseline"])
print("Best adapters and prediction files:", RUN_ROOT)